# 03. Calcul des isochrones

Calcul des isochrones de temps de parcours piéton et cyclable en routage
réel (OSMnx) depuis chaque arrêt de transport en commun structurant, sur
le périmètre de la Métropole Rouen Normandie.

**Prérequis** : exécuter `01_acquisition_donnees_OSM.ipynb` et `02_arrets_TC.ipynb` au préalable  
**Entrées** :
- `data/reseau_pieton_MRN.graphml` — graphe piéton (topologie NetworkX, EPSG:2154)
- `data/reseau_velo_MRN.graphml` — graphe cyclable (topologie NetworkX, EPSG:2154)
- `data/arrets_2026.gpkg` — arrêts TC structurants (situation 2026)

**Constantes** (référence : document de cadrage) :
- Vitesse à pied : 5 km/h
- Vitesse à vélo : 15 km/h
- Seuil d'accessibilité : 15 minutes
- Pas des tranches : 5 minutes

**Projection** : EPSG:2154 (RGF93 / Lambert-93)  
**Sortie** :
- `data/isochrones_2026.gpkg` — polygones d'isochrones dissous par (mode de déplacement × niveau de desserte × tranche 5/10/15 min), livrable cartographique
- `data/acces_noeuds_2026.gpkg` — table du meilleur niveau de desserte atteignable par nœud x niveau du réseau (avec temps de parcours associé), entrée du rattachement des carreaux en `05`

## 1. Imports et paramètres

In [ ]:
from pathlib import Path

import geopandas as gpd
import networkx as nx
import osmnx as ox

print(f"OSMnx version : {ox.__version__}")

# Horizon traité — sélectionne la couche d'arrêts en entrée et nomme les sorties.
# Exécuter le notebook pour chacune des deux valeurs.
HORIZON = "2026"  # "2026" (situation actuelle) | "SERM" (réseau cible)

# Encodage ordinal de la portée structurante (bac exclu). Réutilisé en 05/06
# « Car express » n'apparaît qu'au réseau cible SERM (absent de la situation 2026).
NIVEAUX = {"Bus": 1, "TEOR": 2, "Métro": 3, "Car express": 4, "Train": 5}

# Vitesses de déplacement (km/h) — référence Cerema / INSEE et cycliste urbain ordinaire
VITESSE_PIED = 5
VITESSE_VELO = 15

# Paramètres d'accessibilité (minutes)
SEUIL_ACCESSIBILITE = 15
PAS_TRANCHE = 5

SEUIL_SNAP = 100  # m, aligné sur l'imprécision du carroyage (200 m donc 0–100 m)
TAMPON_ISO = 50  # m, demi-largeur du corridor desservi autour des voies atteintes, impact esthétique uniquement

CRS_PROJET = "EPSG:2154"

Détection de la racine du projet via le fichier sentinelle `.projectroot`,
puis ancrage de tous les chemins dessus. Cette approche est indépendante du
répertoire de travail : elle fonctionne sous JupyterLab comme sous VSCode,
quel que soit le réglage `notebookFileRoot`, et après un clone comme après un
téléchargement ZIP du dépôt.

In [ ]:
def trouver_racine(marqueur=".projectroot"):
    """Remonte depuis le cwd jusqu'au dossier contenant `marqueur`."""
    base = Path.cwd().resolve()
    for parent in (base, *base.parents):
        if (parent / marqueur).exists():
            return parent
    raise FileNotFoundError(f"Racine introuvable (marqueur {marqueur!r})")


ROOT = trouver_racine()
DATA_DIR = ROOT / "data"
print(f"Racine du projet : {ROOT}")

## 2. Chargement des graphes de réseau

Les graphes piéton et cyclable produits par `01_acquisition_donnees_OSM.ipynb`
sont rechargés depuis leurs fichiers `.graphml`, qui conservent la topologie
nécessaire au calcul des plus courts chemins (étapes suivantes).

In [ ]:
G_pieton = ox.load_graphml(DATA_DIR / "reseau_pieton_MRN.graphml")
G_velo   = ox.load_graphml(DATA_DIR / "reseau_velo_MRN.graphml")

# Plus grande composante connexe : écarte les fragments OSM orphelins sur
# lesquels un arrêt se rattacherait, produisant un faux disque (tampon d'un
# quasi-point) au lieu d'une emprise réseau. Cf. anomalie gare sud-ouest.
G_pieton = ox.truncate.largest_component(G_pieton, strongly=True)
G_velo   = ox.truncate.largest_component(G_velo,   strongly=True)

Contrôle de cohérence : nombre de nœuds/arêtes et système de projection de
chaque graphe. Les deux graphes doivent être projetés en EPSG:2154.

In [ ]:
for nom, G in {"piéton": G_pieton, "vélo": G_velo}.items():
    print(
        f"Réseau {nom:<7} | "
        f"{G.number_of_nodes():>7,} nœuds | "
        f"{G.number_of_edges():>7,} arêtes | "
        f"CRS : {G.graph.get('crs')}"
    )

## 3. Chargement des arrêts de transport en commun

Couche d'arrêts TC structurants de l'horizon sélectionné (`arrets_2026.gpkg` ou
`arrets_SERM.gpkg`), produite par `02_arrets_TC.ipynb`.

In [ ]:
arrets = gpd.read_file(DATA_DIR / f"arrets_{HORIZON}.gpkg")

print(f"Horizon : {HORIZON} | {len(arrets):,} arrêts chargés")
print(f"CRS : {arrets.crs}")
arrets.head()

Contrôle de cohérence : la couche des arrêts doit partager le système de
projection des graphes (EPSG:2154) avant tout rattachement spatial des
arrêts au réseau (étape suivante).

In [ ]:
assert arrets.crs is not None, "La couche des arrêts n'a pas de CRS défini."
assert arrets.crs.to_epsg() == 2154, (
    f"CRS inattendu : {arrets.crs.to_epsg()} (attendu : 2154)"
)
assert (arrets["horizon"] == HORIZON).all(), (
    f"Horizon des données ≠ HORIZON={HORIZON!r} "
    f"(trouvé : {sorted(arrets['horizon'].unique())})"
)
print(f"Arrêts horizon {HORIZON} alignés en EPSG:2154.")

## 4. Pondération temporelle du réseau

Le routage des étapes suivantes raisonne en temps de parcours, non en
distance. Chaque arête des deux graphes reçoit donc un attribut `temps`
(en minutes), dérivé de sa longueur `length` — exprimée en mètres, les
graphes étant projetés en EPSG:2154 — et de la vitesse du mode (§ 1).

La vitesse est tenue constante par mode. On n'emploie pas
`ox.routing.add_edge_speeds`, dont les vitesses par type de voie sont
calibrées pour le trafic motorisé et sans pertinence pour les modes actifs.
L'hypothèse d'une allure uniforme — 5 km/h à pied, 15 km/h à vélo — reste
cohérente avec la granularité de la demande (carroyage 200 m) ; ses
simplifications (relief, temps d'attente aux carrefours non modélisés)
relèvent des Limites du projet.

Conversion appliquée à chaque arête :
`temps (min) = length (m) / (vitesse (km/h) × 1000 / 60)`,
soit 83,33 m/min à pied et 250 m/min à vélo.

In [ ]:
# Association graphe - vitesse, identique aux deux horizons (situation 2026 et réseau cible SERM)
RESEAUX = {
    "piéton": (G_pieton, VITESSE_PIED),
    "vélo":   (G_velo,   VITESSE_VELO),
}

for nom, (G, vitesse_kmh) in RESEAUX.items():
    metres_par_min = vitesse_kmh * 1000 / 60
    nx.set_edge_attributes(
        G,
        {
            (u, v, k): donnees["length"] / metres_par_min
            for u, v, k, donnees in G.edges(keys=True, data=True)
        },
        "temps",
    )
    print(f"{nom:<7} : {vitesse_kmh} km/h → {metres_par_min:6.2f} m/min")

Contrôle de cohérence : chaque arête doit porter un temps fini et positif.
La durée de franchissement de l'arête la plus longue donne un ordre de grandeur. 
Un temps anormalement élevé signalerait une arête aberrante (géométrie erronée ou tronçon non découpé) à investiguer avant le routage.

In [ ]:
for nom, (G, _) in RESEAUX.items():
    temps = [donnees["temps"] for *_, donnees in G.edges(data=True)]
    assert len(temps) == G.number_of_edges(), f"Arêtes sans attribut temps ({nom})"
    assert all(t >= 0 for t in temps), f"Temps négatif détecté ({nom})"
    print(
        f"Réseau {nom:<7} | {len(temps):>7,} arêtes pondérées | "
        f"arête la plus longue : {max(temps):.2f} min"
    )

## 5. Rattachement des arrêts au réseau

Le routage part de nœuds du graphe, chaque arrêt est donc rattaché au nœud le plus proche de chaque réseau — un nœud d'origine piéton et un nœud d'origine cyclable, les deux graphes étant distincts.
Le rattachement s'appuie sur la distance euclidienne (graphes et arrêts partagent une projection métrique EPSG:2154, contrôlée au § 3).

`nearest_nodes(..., return_dist=True)` restitue, outre le nœud, la **distance de rattachement** : l'écart entre l'arrêt et son point d'accroche sur le réseau.
Cette distance n'est pas qu'un sous-produit, elle quantifie le biais du rattachement au nœud (*snap-to-node*).
Le voyageur est supposé démarrer au nœud et non à l'arrêt ; ce tronçon d'approche n'est ni routé ni décompté du budget-temps.
Tant qu'il reste petit devant le rayon d'atteinte (1 250 m à pied, 3 750 m à vélo), le biais est négligeable et du même ordre que l'imprécision déjà admise côté demande (carroyage 200 m, soit 0 à 1,2 min).

L'enjeu se concentre sur les modes structurants (TEOR, métro, train, et car express au réseau cible SERM).
Ce sont eux qui portent l'indicateur, et ce sont eux qui peuvent se situer en zone peu maillée — là où le nœud le plus proche est loin, comme l'a laissé entrevoir l'arête de ~3,3 km repérée au § 4.
La distribution des distances de rattachement, lue séparément pour ces modes, décide donc si le snap-to-node simple est défendable ou s'il faut adopter un snap-to-edge.

In [ ]:
SUFFIXE = {"piéton": "pieton", "vélo": "velo"}

X = arrets.geometry.x.to_numpy()
Y = arrets.geometry.y.to_numpy()

for nom, (G, _) in RESEAUX.items():
    noeuds, dists = ox.distance.nearest_nodes(G, X, Y, return_dist=True)
    arrets[f"noeud_{SUFFIXE[nom]}"] = noeuds
    arrets[f"dist_{SUFFIXE[nom]}"] = dists

print("Arrêts rattachés aux deux réseaux.")
arrets[["mode", "noeud_pieton", "dist_pieton", "noeud_velo", "dist_velo"]].head()

Audit des distances de rattachement, par mode et par réseau.
La lecture décisive porte sur les modes structurants (hors bus, qui forme le maillage dense, et hors bac, exclu de la hiérarchie) : leur p95 et leur maximum indiquent si quelques arrêts déterminants s'accrochent loin du réseau.
Le seuil de tolérance est fixé à 100 m — ordre de grandeur de l'imprécision déjà admise côté demande — et la décision se lit sur le 95ᵉ centile, robuste à un arrêt isolé aberrant.

In [ ]:
structurants = ~arrets["mode"].isin(["Bus", "Bac"])

for nom in RESEAUX:
    col = f"dist_{SUFFIXE[nom]}"
    metres_min = (VITESSE_PIED if nom == "piéton" else VITESSE_VELO) * 1000 / 60

    recap = (
        arrets.groupby("mode")[col]
        .agg(n="size", mediane="median", p95=lambda s: s.quantile(0.95), maxi="max")
        .round(1)
    )
    s = arrets.loc[structurants, col]
    verdict = "OK" if s.quantile(0.95) <= SEUIL_SNAP else "À CORRIGER"

    print(f"\n— Réseau {nom} — distance de rattachement (m) —")
    print(recap)
    print(
        f"  Modes structurants : médiane {s.median():.1f} m | "
        f"p95 {s.quantile(0.95):.1f} m ({s.quantile(0.95) / metres_min:.1f} min) | "
        f"max {s.max():.1f} m ({s.max() / metres_min:.1f} min)"
    )
    print(f"  → snap-to-node : {verdict} (seuil p95 = {SEUIL_SNAP} m)")

In [ ]:
# identification de l'arrêt se trouvant à 459,5 m de toute voie accessible aux piétons (a fortiori d'une voie carrossable) 

col = "dist_pieton"
suspect = arrets.loc[arrets[col].idxmax()]
print(suspect[["nom", "mode", "lignes", "operateur", "id_source"]])
print(f"  dist pieton {suspect['dist_pieton']:.1f} m | dist velo {suspect['dist_velo']:.1f} m")
print(f"  X={suspect.geometry.x:.1f}  Y={suspect.geometry.y:.1f}  (EPSG:2154)")

Dans QGIS, l'arrêt nommé "Le Conihout 1183" est dans la presqu'île du Mesnil-sous-Jumièges, sur une arrêt de graphe piéton et vélo, entourée de deux autres arrêts sur la même arrête (les arrêts La Conihout 1827 et Le Conihout 909).
C'est le biais intrinsèque du snap-to-node dans son cas le plus pur : l'arrêt tombe au milieu d'une longue arête, et nearest_nodes ne peut le rattacher qu'à une extrémité. Si l'arête fait ~900 m sans nœud intermédiaire et que l'arrêt est en son milieu, le nœud le plus proche est mécaniquement à environ 450 m alors que le réseau passe à quelques mètres sous l'arrêt.

## 6. Accessibilité par arrêt

Pour chaque arrêt, on détermine les nœuds du réseau atteignables sous le seuil de 15 min depuis son nœud de rattachement (§ 5), avec leur tranche de 5 min.
L'atteignabilité ne dépend que du couple (graphe, nœud d'origine) : elle est indépendante du niveau de desserte, qui est un attribut de l'arrêt rapporté plus tard (§ 7).
Le calcul est donc mené une fois par **nœud d'origine distinct** — plusieurs quais d'un même arrêt partagent souvent leur point de rattachement —, ce qui évite de recalculer des plus courts chemins identiques.

**Méthode : un unique Dijkstra à seuil par origine.**
`single_source_dijkstra_path_length(G, origine, cutoff=15, weight="temps")` restitue en une passe le temps de parcours à tous les nœuds atteignables en ≤ 15 min.
Les tranches 5 / 10 / 15 s'en déduisent par arrondi supérieur : un nœud atteint à *t* minutes appartient à la tranche `5 × ⌈t / 5⌉`, ce qui produit directement des bandes emboîtées sans repasser sur le graphe.

**Pourquoi pas des ego-graphs successifs.**
`nx.ego_graph(G, origine, radius=r, distance="temps")` exécute en interne un Dijkstra à seuil jusqu'à `r`, puis matérialise le sous-graphe correspondant.
Obtenir les trois bandes imposerait trois appels (r = 5, 10, 15), donc trois Dijkstra par origine, recalculant à chaque fois la zone 0–5 puis 0–10 déjà parcourue — environ trois fois plus de calcul de plus courts chemins. S'y ajoute la construction de trois sous-graphes (copies de nœuds et d'arêtes) dont on n'a pas l'usage : seul le dictionnaire `nœud -> temps` est nécessaire pour buffériser les zones atteintes en § 7.
Les deux approches reposent sur le même Dijkstra et donnent le **même résultat**.
Le choix du seuil unique relève de l'efficacité et de la lisibilité, non de la justesse.

In [ ]:
import math

def atteignabilite(G, noeud_origine, cutoff):
    """Temps d'accès continu (minutes) de chaque nœud atteignable depuis
    `noeud_origine`, via un unique Dijkstra à seuil. La conversion en tranche
    (5 / 10 / 15) est dérivée en aval par `tranche_de` : le temps continu —
    nécessaire à la correction du tronçon d'approche en nb05 — n'est plus perdu."""
    return nx.single_source_dijkstra_path_length(
        G, noeud_origine, cutoff=cutoff, weight="temps"
    )


def tranche_de(t, pas=PAS_TRANCHE):
    """Tranche (multiple de `pas`) couvrant le temps continu `t`.
    L'origine (t = 0) est classée en première tranche."""
    return max(pas, pas * math.ceil(t / pas))


acces = {}  # acces[mode][nœud_origine] = {nœud_atteint: temps_continu_min}
for nom, (G, _) in RESEAUX.items():
    col = f"noeud_{SUFFIXE[nom]}"
    origines = arrets[col].unique()
    acces[nom] = {
        orig: atteignabilite(G, orig, SEUIL_ACCESSIBILITE)
        for orig in origines
    }
    couples = sum(len(r) for r in acces[nom].values())
    print(
        f"Réseau {nom:<7} | {len(origines):>5,} nœuds d'origine distincts | "
        f"{couples:>10,} nœuds atteints (cumul tous arrêts)"
    )

In [ ]:
SEUIL_FRAGMENT = 10  # informational : en deçà, isochrone confinée à inspecter

for nom in RESEAUX:
    col = f"noeud_{SUFFIXE[nom]}"
    suspects = {o: len(a) for o, a in acces[nom].items() if len(a) < SEUIL_FRAGMENT}
    if not suspects:
        print(f"— {nom} : aucun —\n")
        continue
    print(f"— {nom} : {len(suspects)} origine(s) sous {SEUIL_FRAGMENT} nœuds —")
    sub = arrets[arrets[col].isin(suspects)].copy()
    sub["n_atteints"] = sub[col].map(suspects)
    print(sub[["nom", "mode", f"dist_{SUFFIXE[nom]}", "n_atteints"]]
          .sort_values("n_atteints").to_string(index=False))
    print()

In [ ]:
for nom in RESEAUX:
    tailles = [len(a) for a in acces[nom].values()]
    n = sum(1 for t in tailles if t < SEUIL_FRAGMENT)
    etat = "OK" if n == 0 else f"{n} origine(s) confinée(s) — voir diagnostic"
    print(f"Réseau {nom:<7} | min nœuds atteints : {min(tailles):>4} | {etat}")

Quelques arrêts bus de frange à atteinte piétonne confinée, isolement réel et non fragment, sans effet sur l'indicateur structurant

Contrôle de cohérence sur un arrêt témoin d'un mode structurant : les nœuds atteints doivent se répartir sur les seules tranches du gradient (5 / 10 / 15), sans valeur hors borne. La tranche reflète la *première* atteinte ; le compte par tranche correspond donc à des anneaux successifs, non à un cumul.

In [ ]:
import pandas as pd

temoin = arrets.loc[arrets["mode"] == "Métro"].iloc[0]
bandes = (
    pd.Series(acces["piéton"][temoin["noeud_pieton"]])
    .map(tranche_de)
    .value_counts()
    .sort_index()
)

print(f"Arrêt témoin : {temoin['nom']} ({temoin['mode']})")
print("Nœuds atteignables à pied, par tranche (min) :")
print(bandes.to_string())

assert set(bandes.index) <= {PAS_TRANCHE, 2 * PAS_TRANCHE, SEUIL_ACCESSIBILITE}, \
    "Tranche hors gradient 5 / 10 / 15"
print("\nTranches conformes au gradient 5 / 10 / 15.")

## 7. Construction des isochrones (polygones)

Les nœuds atteignables (§ 6) sont convertis en zones surfaciques.
Chaque arête dont les deux extrémités sont atteintes sous la tranche considérée est **tamponnée** d'une demi-largeur de corridor, puis l'ensemble est fusionné (`union_all`).
On écarte l'enveloppe convexe, qui surestimerait grossièrement l'emprise en péri-urbain : le tampon de réseau épouse les voies réellement parcourues.
Les tranches étant emboîtées (un nœud atteint à ≤ 5 min l'est aussi à ≤ 10 et ≤ 15), les polygones le sont également - utile en cartographie comme au croisement avec la population (§ 05).

Le **niveau de desserte** de l'arrêt est conservé sur chaque polygone, et doublé d'un **encodage ordinal** (`niveau_ordre`, entier) traduisant la portée structurante `bus < TEOR < métro < car express < train`. Le bac, hors hiérarchie, est exclu.
Le niveau « aucun » (carreau couvert par aucune isochrone) n'est pas matérialisé, c'est l'absence.

La sortie est **dissoute par (mode de déplacement × niveau × tranche)** : une seule emprise par triplet, plutôt que des milliers d'isochrones par arrêt qui se recouvriraient. L'identité de l'arrêt est perdue, mais aucun traitement aval ne s'en sert (§ 05 raisonne en niveaux, non en arrêts).
La dissolution est faite au niveau des nœuds - union des nœuds atteints par niveau avant tamponnage, ce qui donne le même résultat qu'une dissolution de polygones, en bien moins d'unions.

In [ ]:
TRANCHES = list(range(PAS_TRANCHE, SEUIL_ACCESSIBILITE + 1, PAS_TRANCHE))  # [5, 10, 15]
# HORIZON est fixé en tête de notebook (§1) et contrôlé au chargement (§3).

# Arêtes en GeoDataFrame, une fois par réseau (colonnes u, v pour le filtrage)
edges = {
    nom: ox.graph_to_gdfs(G, nodes=False, edges=True).reset_index()
    for nom, (G, _) in RESEAUX.items()
}

arrets_hierarchises = arrets[arrets["mode"].isin(NIVEAUX)]  # exclut le bac

records = []
for nom, (G, _) in RESEAUX.items():
    e = edges[nom]
    col_noeud = f"noeud_{SUFFIXE[nom]}"
    for niveau_lbl, niveau_ord in NIVEAUX.items():
        origines = arrets_hierarchises.loc[arrets_hierarchises["mode"] == niveau_lbl, col_noeud].unique()
        if len(origines) == 0:
            continue  # ex. car express en 2026
        for tranche in TRANCHES:
            noeuds = set()
            for orig in origines:
                noeuds.update(n for n, t in acces[nom][orig].items() if t <= tranche)
            sel = e[e["u"].isin(noeuds) & e["v"].isin(noeuds)]
            if sel.empty:
                continue
            records.append({
                "mode_deplacement": nom,
                "niveau": niveau_lbl,
                "niveau_ordre": niveau_ord,
                "tranche_min": tranche,
                "horizon": HORIZON,
                "geometry": sel.geometry.buffer(TAMPON_ISO).union_all(),
            })

isochrones = gpd.GeoDataFrame(records, crs=CRS_PROJET)
print(f"{len(isochrones)} isochrones dissoutes (horizon {HORIZON})")

24 (isochrones) = 2 (modes doux) x 4 (modes TC) x 3 (tranches)

Contrôle de cohérence : aire de chaque emprise par niveau et par tranche.
Deux attentes, l'emboîtement (l'aire croît avec la tranche pour un niveau donné : 5 ≤ 10 ≤ 15) et la validité géométrique.
La comparaison *entre* niveaux n'est en revanche pas monotone : un niveau de portée supérieure (train) compte moins d'arrêts qu'un niveau inférieur (bus) et couvre donc une emprise plus petite, c'est attendu, et c'est précisément ce que mesure l'indicateur.

In [ ]:
assert isochrones.geometry.is_valid.all(), "Géométrie(s) invalide(s)"

recap = (
    isochrones.assign(aire_km2=isochrones.geometry.area / 1e6)
    .pivot_table(index=["mode_deplacement", "niveau_ordre", "niveau"],
                 columns="tranche_min", values="aire_km2")
    .round(1)
    .sort_index()
)
print("Aire des emprises (km²) par niveau et tranche :")
print(recap.to_string())

# Emboîtement : pour chaque (mode, niveau), l'aire doit croître avec la tranche
croissant = recap.apply(lambda r: r.dropna().is_monotonic_increasing, axis=1)
assert croissant.all(), "Emboîtement rompu (aire non croissante avec la tranche)"
print("\nEmboîtement des tranches vérifié (5 ≤ 10 ≤ 15).")

Le contrôle est OK

**L'emboîtement par tranche** est vérifié, et les ratios entre tranches sont plausibles. À pied, niveau bus : 166 < 276 < 358 km², soit des incréments décroissants (×1,66 puis ×1,30). C'est la signature attendue d'isochrones qui saturent — les premières minutes ouvrent du territoire vierge, les dernières recouvrent des zones déjà atteintes par des arrêts voisins.

**Le ratio vélo/piéton** est cohérent avec le §6, et il varie intelligemment selon le niveau. Sur le bus à 15 min, vélo/piéton = 576/358 ≈ 1,6 — modeste, parce que le maillage bus est dense et déjà presque saturé à pied, le vélo n'ajoute pas tant de territoire neuf. Sur le train à 15 min, le ratio explose : 115/20 ≈ 5,8. Logique et porteur de sens : les gares sont peu nombreuses et isolées, leurs isochrones piétonnes ne se recouvrent pas, donc passer à vélo (rayon ×3) ouvre du territoire pur sans redondance. Plus le mode est rare et dispersé, plus le vélo démultiplie son emprise. C'est précisément le principe de l'indicateur : le vélo comme rabattement vers les points structurants isolés.

**La hiérarchie des aires** décroît avec la portée, comme annoncé au contrôle du §7. bus > TEOR > métro > train à emprise égale de tranche : le mode le plus structurant couvre le moins de surface, parce qu'il a le moins d'arrêts. Ce n'est pas un défaut, c'est l'objet même de l'analyse, et c'est ce qui rend le « meilleur niveau atteignable » discriminant. Autre remarque : à pied, train (20 km²) < métro (26,5) à 15 min, mais à vélo train (115) > métro (66). L'inversion vient de la géographie, les gares TER rayonnent sur un territoire large et péri-urbain, le métro reste resserré sur l'axe urbain. En vélo, la dispersion des gares devient un avantage de couverture.

## 8. Assemblage et export

Cette étape écrit les deux sorties du notebook, de natures distinctes.

**Isochrones (livrable cartographique).** Les 24 emprises dissoutes (§ 7) sont exportées telles quelles, elles servent la mise en carte.

**Table du meilleur niveau atteignable par nœud (livrable analytique).**
C'est l'entrée du rattachement des carreaux en `05`. Elle est obtenue en *inversant* la structure `acces` (§ 6).
Le déplacement à pied ou à vélo étant symétrique en temps, « l'arrêt atteint le nœud R en t minutes » équivaut à « depuis R, on
atteint l'arrêt en t minutes » : un carreau rattaché près de R accède donc à la desserte de cet arrêt en t. Le niveau étant porté par l'arrêt (et non par le nœud d'origine), l'inversion parcourt les nœuds d'origine en leur associant le ou les niveaux des arrêts qui s'y rattachent, puis propage à chaque nœud atteint.

La table est tenue **par nœud × niveau** (et non « meilleur niveau seulement ») : pour chaque nœud, on conserve le temps minimal d'accès à *chaque* niveau qu'il
atteint. Cette granularité permettra à `05` de restituer non seulement le meilleur niveau, mais le temps d'accès à chacun — utile aux histogrammes par tranche prévus au cadrage. La déduplication retient, par (réseau, nœud, niveau), la tranche minimale. Elle est menée sur les seuls nœuds d'origine distincts (§ 6), un même nœud pouvant porter plusieurs arrêts.

In [ ]:
# INVERSION-DEDUPLICATION

from collections import defaultdict

LIBELLE = {ordre: lbl for lbl, ordre in NIVEAUX.items()}  # 1 → "Bus", …

# (réseau, nœud atteint, niveau_ordre) → temps d'accès minimal (continu, minutes)
meilleur = {}
for nom, (G, _) in RESEAUX.items():
    col_noeud = f"noeud_{SUFFIXE[nom]}"

    # Niveaux desservis par chaque nœud d'origine (un nœud peut porter
    # plusieurs arrêts, éventuellement de niveaux différents)
    niveaux_par_origine = defaultdict(set)
    for orig, mode_arret in zip(arrets_hierarchises[col_noeud],
                                arrets_hierarchises["mode"]):
        niveaux_par_origine[orig].add(NIVEAUX[mode_arret])

    # Propagation à chaque nœud atteint depuis ces origines
    for orig, niveaux in niveaux_par_origine.items():
        for node, t in acces[nom][orig].items():
            for niveau_ord in niveaux:
                cle = (nom, node, niveau_ord)
                if t < meilleur.get(cle, float("inf")):
                    meilleur[cle] = t

print(f"{len(meilleur):,} couples (réseau × nœud × niveau)")

Le chiffre de 238 997 couples est cohérent et recoupé avec le §6.
Au §6 nous avions ~1,49 M (piéton) + 5,94 M (vélo) ≈ 7,4 M couples (nœud atteint, arrêt) cumulés, soit ~30× plus que la table.
C'est exactement le sens de l'inversion-déduplication : on passe de « chaque nœud compté autant de fois qu'il y a d'arrêts qui l'atteignent » à « chaque nœud compté une fois par niveau distinct ». Avec 2 854 arrêts bus pour 4 niveaux, un nœud en zone dense était atteint par des dizaines d'arrêts bus — tous écrasés en une seule ligne (réseau, nœud, bus, tranche min). Une compression d'un ordre de grandeur est précisément ce qu'on attend ; un chiffre resté proche du million aurait signalé que la déduplication ne mordait pas.
L'ordre de grandeur absolu est sain aussi. Les graphes totalisent ~107 k nœuds (61 k piéton + 46 k vélo). 234 619 couples / ~107 k nœuds ≈ 2,2 niveaux par nœud en moyenne — autrement dit, un nœud typique atteint un peu plus de deux niveaux de desserte sous 15 min (souvent bus + TEOR, parfois métro). Plausible en agglomération : le maillage bus couvre presque partout, les modes structurants s'y ajoutent localement. Si la moyenne avait été proche de 1, on se serait demandé où passaient les superpositions ; proche de 4, on aurait soupçonné un sur-comptage.

In [ ]:
import pandas as pd

acces_table = pd.DataFrame(
    [(nom, node, niv, t) for (nom, node, niv), t in meilleur.items()],
    columns=["mode_deplacement", "node", "niveau_ordre", "_t_min"],
)
# Contrat nb03 → nb05 : temps d'accès continu conservé en secondes (entier) ;
# tranche cartographique 5 / 10 / 15 dérivée du même temps.
acces_table["temps_acces_s"] = (acces_table["_t_min"] * 60).round().astype("int64")
acces_table["tranche_min"]   = acces_table["_t_min"].map(tranche_de).astype("int64")
acces_table = acces_table.drop(columns="_t_min")
acces_table["niveau"]  = acces_table["niveau_ordre"].map(LIBELLE)
acces_table["horizon"] = HORIZON

# Géométrie des nœuds, jointe par réseau (les deux graphes ont des nœuds distincts)
parts = []
for nom, (G, _) in RESEAUX.items():
    noeuds_geo = ox.graph_to_gdfs(G, nodes=True, edges=False)[["geometry"]]
    part = (
        acces_table[acces_table["mode_deplacement"] == nom]
        .merge(noeuds_geo, left_on="node", right_index=True)
    )
    parts.append(part)

acces_noeuds = gpd.GeoDataFrame(
    pd.concat(parts, ignore_index=True), geometry="geometry", crs=CRS_PROJET
)
print(f"{len(acces_noeuds):,} lignes | {acces_noeuds['node'].nunique():,} nœuds distincts")

238 997 lignes pour 82 210 nœuds distincts, soit 2,9 niveaux par nœud en moyenne, un peu plus que l'estimation précédente à 2,2, parce que j'avais rapporté aux ~107 k nœuds totaux des deux graphes, alors que seuls les nœuds effectivement atteints sous 15 min figurent dans la table. C'est la bonne lecture : 82 210 est le nombre de nœuds que la table « éclaire », tous réseaux confondus. Les nœuds jamais atteints (zones hors de portée de tout arrêt) n'y sont pas — c'est voulu, ils correspondront au niveau « aucun » en 05, matérialisé par l'absence.
2,9 niveaux par nœud atteint est cohérent et même un peu rassurant : un nœud typiquement desservi l'est par près de trois niveaux (souvent bus + TEOR + métro, ou bus + TEOR + train selon le secteur). Ça reflète la superposition réelle des réseaux en cœur d'agglomération, là où se concentrent les nœuds. En frange, les nœuds n'atteignent que le bus, voire rien, ce qui tire la moyenne vers le bas, donc la vraie superposition en zone dense est encore supérieure à 2,85. Tout cela est conforme à ce que l'indicateur veut capter.
Un point de prudence sur l'interprétation, pour ne pas sur-lire ce nombre : 82 210 « nœuds distincts » additionne les nœuds piéton et les nœuds vélo, qui sont des objets différents (graphes distincts, osmid propres à chaque réseau). Ce n'est donc pas « 82 210 points du territoire » mais « 82 210 (réseau, nœud) ». Pour une lecture spatiale — combien de lieux distincts sont desservis — il faudrait raisonner par réseau séparément.

Contrôle de cohérence avant export : les tranches restent dans le gradient 5 / 10 / 15, les niveaux dans la hiérarchie, et la jointure géométrique est complète (aucun nœud sans coordonnées). 
On vérifie aussi qu'aucune ligne n'a été perdue au merge.

In [ ]:
assert set(acces_noeuds["tranche_min"]) <= set(TRANCHES), "Tranche hors gradient"
assert set(acces_noeuds["niveau_ordre"]) <= set(NIVEAUX.values()), "Niveau hors hiérarchie"
assert acces_noeuds.geometry.notna().all(), "Nœud(s) sans géométrie après merge"
assert len(acces_noeuds) == len(acces_table), "Lignes perdues au merge géométrique"

# Contrat nb03 → nb05 : temps d'accès continu présent, borné (0 = nœud-arrêt),
# et cohérent avec la borne haute de sa tranche.
SEUIL_S = SEUIL_ACCESSIBILITE * 60
assert acces_noeuds["temps_acces_s"].between(0, SEUIL_S).all(), \
    f"temps_acces_s hors [0, {SEUIL_S}] s"
assert (acces_noeuds["temps_acces_s"] <= acces_noeuds["tranche_min"] * 60).all(), \
    "temps_acces_s incohérent : temps continu au-delà de la borne haute de sa tranche"

# Répartition par niveau (contrôle de vraisemblance : décroît avec la portée)
print("Nœuds atteignant chaque niveau (réseau piéton, ≤ 15 min) :")
apercu = (
    acces_noeuds[acces_noeuds["mode_deplacement"] == "piéton"]
    .groupby(["niveau_ordre", "niveau"])["node"].nunique()
)
print(apercu.to_string())

print("Nœuds atteignant chaque niveau (réseau vélo, ≤ 15 min) :")
apercu = (
    acces_noeuds[acces_noeuds["mode_deplacement"] == "vélo"]
    .groupby(["niveau_ordre", "niveau"])["node"].nunique()
)
print(apercu.to_string())

# Export des deux livrables
sortie_iso = DATA_DIR / f"isochrones_{HORIZON}.gpkg"
sortie_noeuds = DATA_DIR / f"acces_noeuds_{HORIZON}.gpkg"
isochrones.to_file(sortie_iso, driver="GPKG")
acces_noeuds.to_file(sortie_noeuds, driver="GPKG")
print(f"\nÉcrit : {sortie_iso.name} ({len(isochrones)} isochrones)")
print(f"Écrit : {sortie_noeuds.name} ({len(acces_noeuds):,} lignes nœud × niveau)")

PIETON
La décroissance est nette et monotone, l'inversion est validée. bus (70 703) > TEOR (20 463) > métro (12 465) > train (7 204), exactement l'ordre de portée structurante.
Aucun niveau supérieur ne dépasse un niveau inférieur, ce qui confirme que les niveaux n'ont pas été brouillés au moment de propager depuis les arrêts.
Le recoupement avec les aires du §7 est cohérent, et révèle un détail intéressant. À pied, les nœuds décroissent bus → TEOR avec un facteur ~3,5 (70 703 / 20 463), et les aires du §7 faisaient pareil (358 / 45 ≈ 8 — un peu plus marqué, parce que l'aire pénalise davantage la dispersion que le simple comptage de nœuds).
Train compte moins de nœuds que métro (7 204 < 12 465) à pied, ce qui colle avec les aires (train 21 km² < métro 27 km²) : les gares sont peu nombreuses et leurs isochrones piétonnes ne se recouvrent pas. Tout est interne-cohérent entre les deux livrables.

VELO
D'abord la cohérence : à vélo aussi la décroissance tient, bus (61 817) > TEOR (24 201) > métro (16 927)… puis train (25 217) qui repasse au-dessus du métro et du TEOR. Ce n'est pas une anomalie, c'est l'inversion déjà vue dans les aires du §7 (à vélo, train 133 km² > métro 66 km²), et elle s'explique par la même géographie. Les gares TER sont dispersées sur un large territoire péri-urbain, à pied leurs isochrones restent petites et disjointes (d'où train < métro à pied : 7 204 < 12 465), mais à vélo le rayon ×3 ouvre autour de chaque gare un territoire vaste et sans recouvrement, qui finit par éclipser le métro resserré sur l'axe urbain. Les deux livrables racontent la même histoire par deux mesures indépendantes (nœuds et aires) — c'est la meilleure validation qu'on puisse souhaiter.

L'observation porteuse pour l'analyse globale est dans la comparaison pied/vélo par niveau, car elle n'est pas uniforme :
Le bus recule à vélo (70 703 → 61 817). Contre-intuitif au premier abord, mais logique : le graphe vélo a moins de nœuds que le piéton (46 k vs 61 k), et le maillage bus sature déjà à pied. Passer au vélo n'ouvre presque pas de territoire neuf, et la moindre densité de nœuds vélo fait baisser le compte. Le bus est partout, à pied comme à vélo. Le mode de déplacement ne change rien à sa quasi-ubiquité.
À l'inverse, le train multiplie par 3,5 (7 204 → 25 217) et le métro grimpe d'un tiers. Plus le mode est rare et ponctuel, plus le vélo démultiplie son emprise — parce qu'il n'y a pas de recouvrement à « gaspiller ». C'est précisément le mécanisme du rabattement vélo vers les points structurants isolés, et c'est le cœur de ce que le SERM veut activer.

Cette asymétrie est un résultat en soi : elle dit que le vélo est le mode qui rend les gares accessibles, là où la marche suffit déjà pour le bus.
Message clé à restituer pour le SERM : les nouvelles haltes n'ont d'impact qu'à condition d'un rabattement cyclable, puisque à pied leur emprise reste confinée.
A garder sous la main le chiffre (train ×3,5 entre pied et vélo), il objective l'enjeu du stationnement vélo aux haltes que mentionne mon cadrage.

In [ ]:
STRUCTURANTS = ["Train", "Car express", "Métro", "TEOR"]  # bus et bac exclus (hors hiérarchie structurante)

def aire_structurante(iso, mode_dep, tranche):
    sel = iso[(iso["mode_deplacement"] == mode_dep)
              & (iso["tranche_min"] == tranche)
              & (iso["niveau"].isin(STRUCTURANTS))]
    return sel.geometry.union_all().area / 1e6

aire_velo = aire_structurante(isochrones, "vélo",   15)
aire_pied = aire_structurante(isochrones, "piéton", 15)
print(f"vélo   structurant 15 min : {aire_velo:6.1f} km²")
print(f"piéton structurant 15 min : {aire_pied:6.1f} km²")
print(f"ratio vélo / piéton       : {aire_velo / aire_pied:.1f}×")